In [ ]:
# Define the baseline model
Baseline_models={
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=6,  random_state=42),
    "Logistic Regression": LogisticRegression(C=1.0, max_iter=1000,  random_state=42),
}

In [ ]:
#Train and Evaluate Models

results = []

for name, model in Baseline_models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    prob = pipeline.predict_proba(X_test)[:,1]
    acc = accuracy_score(y_test, y_pred)
    rec=recall_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, prob)

    print(f"\nModel: {name}")
    print(f"Accuracy: {acc}")
    print(f"Recall: {rec}")
    print(f"ROC_AUC: {roc_auc}\n")
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='coolwarm')
    plt.title(f"Confusion Matrix - {name}")
    plt.show()

    results.append((name, acc))

In [ ]:
# Calculate class weight ratio for imbalance handling
num_neg = (y_train == 0).sum()
num_pos = (y_train == 1).sum()
scale_pos_weight = num_neg / num_pos # Resolves class imbalance
class_weight = {0: 1, 1: scale_pos_weight}
print(f"Class weight ratio: {scale_pos_weight:.2f}")
print(f"Class weights: {class_weight}")

In [ ]:
#Define candidate models

candidates = {
    "XGBoost": XGBClassifier(n_estimators=400, max_depth=10, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,scale_pos_weight=scale_pos_weight, eval_metric="logloss", random_state=42),
    "LightGBM": LGBMClassifier(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1),
}

In [ ]:
#Train and Evaluate Models

results = []

for name, model in candidates.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    prob = pipeline.predict_proba(X_test)[:,1]
    acc = accuracy_score(y_test, y_pred)
    rec=recall_score(y_test, y_pred)
    roc_auc= roc_auc_score(y_test, prob)

    print(f"\nModel: {name}")
    print(f"Accuracy: {acc}")
    print(f"Recall: {rec}")
    print(f"ROC_AUC: {roc_auc}\n")
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='coolwarm')
    plt.title(f"Confusion Matrix - {name}")
    plt.show()

    results.append((name, acc))

In [ ]:

# Bumping up the scale_pos_weight beyond the natural 2.77 ratio
tuned_weight =  scale_pos_weight #* 2.5


final_model = {
    "tuned_XGBoost": XGBClassifier(n_estimators=500,
                                   max_depth=4,
                                   learning_rate=0.03,
                                   subsample=0.8,
                                   scale_pos_weight=tuned_weight,
                                   min_child_weight=4,
                                   colsample_bytree=0.8,
                                   reg_alpha=1.0,  # L1 regularization (penalizes complex weights)
                                   reg_lambda=1.0,  # L2 regularization (smooths out predictions)
                                   gamma=0.3,  # Minimum loss reduction required to make a split
                                   eval_metric="logloss",
                                   random_state=42),}

In [ ]:
#Train and Evaluate Models

results = []

for name, model in final_model.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    prob = pipeline.predict_proba(X_test)[:,1]
    acc = accuracy_score(y_test, y_pred)
    rec=recall_score(y_test, y_pred)
    roc_auc= roc_auc_score(y_test, prob)

    print(f"\nModel: {name}")
    print(f"Accuracy: {acc}")
    print(f"Recall: {rec}")
    print(f"ROC_AUC: {roc_auc}\n")
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='coolwarm')
    plt.title(f"Confusion Matrix - {name}")
    plt.show()

    results.append((name, acc))